# Genie Space Conversation Duration Logger

Incrementally captures Genie space conversations and logs per-message latency with a three-stage breakdown:

| Stage | What it measures | Source |
| --- | --- | --- |
| **pre_execution** | AI generation, intent classification, SQL planning — before SQL hits the warehouse | `question_timestamp` → `system.query.history.start_time` |
| **query_execution** | Actual SQL execution on the warehouse | `system.query.history.execution_duration_ms` |
| **post_execution** | Result processing + LLM summarization after SQL completes | `system.query.history.end_time` → `response_timestamp` |

`total_duration = pre_execution + query_execution + post_execution`

### How it works

1. Fetches all conversations & messages from the Genie REST API (parallelized, 8 workers)
2. Joins `system.query.history` (1-year retention) for SQL timing breakdown
3. Joins `system.access.audit` for conversation mode (`chat` / `agent`)
4. MERGEs results into a Delta table keyed on `(space_id, conversation_id, message_id)`

### Parameters

| Widget | Purpose |
| --- | --- |
| `space_id` | Genie space to pull conversations from |
| `catalog_name` | Target Unity Catalog catalog |
| `schema_name` | Target schema |
| `table_name` | Target table  |

### Understanding NULL values in the timing breakdown

| Scenario | Breakdown available? | Explanation |
| --- | --- | --- |
| `sql_query` response, within 1 year | Yes — all three stages populated | Joined with `system.query.history` for precise timing |
| `sql_query` response, older than 1 year | No — stages are NULL | System table retention exceeded; only `total_duration_sec` is available |
| `text_response` (no SQL executed) | Yes — `pre_execution = total`, query/post = 0 | Entire duration is AI processing, no SQL was involved |

### How `conversation_mode` is determined

The audit log records `conversation_type` only when a conversation is created via the explicit `createConversation` action (`NORMAL` → `chat`, `DEEP_RESEARCH` → `agent`). Conversations started by typing directly into the Genie input use the `genieStartConversationMessage` path, which doesn’t log the mode.

**Assumption applied:** NULLs default to `'chat'` because Deep Research requires an explicit opt-in that always routes through `createConversation` (which IS logged). The implicit input path is always normal chat mode.

### Scheduling recommendation

Run this notebook regularly (e.g. daily) so timing data is captured and persisted before `system.query.history` retention (1 year) expires.

In [0]:
# Parameters
dbutils.widgets.text("space_id", "", "Genie Space ID")
dbutils.widgets.text("catalog_name", "", "Target Catalog")
dbutils.widgets.text("schema_name", "", "Target Schema")
dbutils.widgets.text("table_name", "", "Target Table Name")

SPACE_ID = dbutils.widgets.get("space_id")
CATALOG = dbutils.widgets.get("catalog_name")
SCHEMA = dbutils.widgets.get("schema_name")
TABLE_NAME = dbutils.widgets.get("table_name")
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"Space ID: {SPACE_ID}")
print(f"Target table: {FULL_TABLE_NAME}")

In [0]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Workspace context for API calls
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def api_get(path: str, params: dict | None = None) -> dict:
    """GET request to the Databricks REST API with retry on 429."""
    url = f"{HOST}{path}"
    for attempt in range(3):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if resp.status_code == 429:
            import time
            time.sleep(2 ** attempt)
            continue
        resp.raise_for_status()
        return resp.json()
    resp.raise_for_status()
    return resp.json()


print(f"Connected to: {HOST}")

In [0]:
def list_conversations(space_id: str) -> list[dict]:
    """Retrieve all conversations for a Genie space with pagination."""
    conversations, page_token = [], None
    while True:
        params = {"page_size": 100}
        if page_token:
            params["page_token"] = page_token
        data = api_get(f"/api/2.0/genie/spaces/{space_id}/conversations", params=params)
        convs = data.get("conversations", [])
        conversations.extend(convs)
        page_token = data.get("next_page_token")
        if not page_token or not convs:
            break
    return conversations


def list_messages(space_id: str, conversation_id: str) -> list[dict]:
    """Retrieve all messages for a given conversation with pagination."""
    messages, page_token = [], None
    while True:
        params = {"page_size": 100}
        if page_token:
            params["page_token"] = page_token
        data = api_get(
            f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages",
            params=params,
        )
        msgs = data.get("messages", [])
        messages.extend(msgs)
        page_token = data.get("next_page_token")
        if not page_token or not msgs:
            break
    return messages


conversations = list_conversations(SPACE_ID)
print(f"Found {len(conversations)} conversations")

In [0]:
def classify_response_type(msg: dict) -> str:
    """Classify response type: sql_query, text_response, error, or in_progress."""
    if msg.get("status") == "EXECUTING_QUERY":
        return "in_progress"
    if msg.get("error"):
        return "error"
    for att in (msg.get("attachments") or []):
        if att.get("query", {}).get("statement_id"):
            return "sql_query"
    return "text_response"


def _fetch_conversation_records(conv: dict) -> list[dict]:
    """Fetch messages for one conversation and return parsed records."""
    conversation_id = conv["conversation_id"]
    conv_title = conv.get("title", "")
    records = []

    try:
        messages = list_messages(SPACE_ID, conversation_id)
    except requests.HTTPError:
        return records

    for msg in messages:
        created_ts = msg.get("created_timestamp", 0)
        updated_ts = msg.get("last_updated_timestamp", 0)
        response_type = classify_response_type(msg)

        total_duration_sec = (updated_ts - created_ts) / 1000.0 if updated_ts > created_ts else None

        # Extract statement_id from attachments
        sql_statement_id, generated_sql = None, None
        for att in (msg.get("attachments") or []):
            qi = att.get("query", {})
            if qi.get("statement_id"):
                sql_statement_id = qi["statement_id"]
                generated_sql = qi.get("query", "")[:2000]
                break

        # Text responses: all time is AI processing
        pre_execution_sec = query_execution_sec = post_execution_sec = None
        if response_type == "text_response" and total_duration_sec is not None:
            pre_execution_sec, query_execution_sec, post_execution_sec = total_duration_sec, 0.0, 0.0

        records.append({
            "space_id": SPACE_ID,
            "conversation_id": conversation_id,
            "conversation_title": conv_title,
            "message_id": msg.get("message_id", ""),
            "user_id": msg.get("user_id"),
            "user_question": (msg.get("content") or "")[:500] or None,
            "status": msg.get("status", ""),
            "response_type": response_type,
            "question_timestamp": created_ts,
            "response_timestamp": updated_ts if updated_ts > created_ts else None,
            "total_duration_sec": round(total_duration_sec, 3) if total_duration_sec is not None else None,
            "pre_execution_sec": round(pre_execution_sec, 3) if pre_execution_sec is not None else None,
            "query_execution_sec": round(query_execution_sec, 3) if query_execution_sec is not None else None,
            "post_execution_sec": round(post_execution_sec, 3) if post_execution_sec is not None else None,
            "sql_statement_id": sql_statement_id,
            "generated_sql": generated_sql,
        })
    return records


# Parallel fetch: 8 workers avoids rate-limiting while being ~4x faster than sequential
print(f"Fetching messages from {len(conversations)} conversations (parallel)...")
timing_records = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(_fetch_conversation_records, c): c for c in conversations}
    for future in as_completed(futures):
        timing_records.extend(future.result())

print(f"Extracted {len(timing_records)} records "
      f"({sum(1 for r in timing_records if r['sql_statement_id'])} SQL, "
      f"{sum(1 for r in timing_records if r['response_type'] == 'text_response')} text)")

In [0]:
# Enrich records with:
# 1. Query execution timing from system.query.history (1-year retention)
# 2. Conversation mode (chat/agent) from system.access.audit

SCHEMA_DEF = StructType([
    StructField("space_id", StringType(), False),
    StructField("conversation_id", StringType(), False),
    StructField("conversation_title", StringType(), True),
    StructField("message_id", StringType(), False),
    StructField("user_id", LongType(), True),
    StructField("user_question", StringType(), True),
    StructField("status", StringType(), True),
    StructField("response_type", StringType(), True),
    StructField("question_timestamp", LongType(), True),
    StructField("response_timestamp", LongType(), True),
    StructField("total_duration_sec", DoubleType(), True),
    StructField("pre_execution_sec", DoubleType(), True),
    StructField("query_execution_sec", DoubleType(), True),
    StructField("post_execution_sec", DoubleType(), True),
    StructField("sql_statement_id", StringType(), True),
    StructField("generated_sql", StringType(), True),
])

if not timing_records:
    print("No timing records extracted.")
    df_new = spark.createDataFrame([], schema=SCHEMA_DEF)
else:
    df_base = spark.createDataFrame(timing_records, schema=SCHEMA_DEF)
    stmt_ids = [r["sql_statement_id"] for r in timing_records if r["sql_statement_id"]]

    # --- Join query timing from system.query.history ---
    if stmt_ids:
        spark.createDataFrame([(s,) for s in stmt_ids], ["stmt_id"]).createOrReplaceTempView("_genie_stmt_ids")
        df_qh = spark.sql("""
            SELECT statement_id,
                   BIGINT(UNIX_TIMESTAMP(start_time) * 1000) AS query_start_ms,
                   BIGINT(UNIX_TIMESTAMP(end_time) * 1000) AS query_end_ms,
                   execution_duration_ms
            FROM system.query.history
            INNER JOIN _genie_stmt_ids ON statement_id = stmt_id
        """)
        df_new = (
            df_base.alias("b")
            .join(df_qh.alias("qh"), F.col("b.sql_statement_id") == F.col("qh.statement_id"), "left")
            .withColumn("pre_execution_sec", F.when(
                F.col("b.response_type") == "sql_query",
                F.round((F.col("qh.query_start_ms") - F.col("b.question_timestamp")) / 1000.0, 3)
            ).otherwise(F.col("b.pre_execution_sec")))
            .withColumn("query_execution_sec", F.when(
                F.col("b.response_type") == "sql_query",
                F.round(F.col("qh.execution_duration_ms") / 1000.0, 3)
            ).otherwise(F.col("b.query_execution_sec")))
            .withColumn("post_execution_sec", F.when(
                F.col("b.response_type") == "sql_query",
                F.round((F.col("b.response_timestamp") - F.col("qh.query_end_ms")) / 1000.0, 3)
            ).otherwise(F.col("b.post_execution_sec")))
            .select(
                "b.space_id", "b.conversation_id", "b.conversation_title", "b.message_id",
                "b.user_id", "b.user_question", "b.status", "b.response_type",
                "b.question_timestamp", "b.response_timestamp", "b.total_duration_sec",
                "pre_execution_sec", "query_execution_sec", "post_execution_sec",
                "b.sql_statement_id", "b.generated_sql"
            )
        )
    else:
        df_new = df_base

    # --- Join conversation mode from audit log (scoped to actual date range) ---
    min_ts = min(r["question_timestamp"] for r in timing_records if r["question_timestamp"]) // 1000
    from datetime import date, timedelta
    earliest_date = date.fromtimestamp(min_ts) - timedelta(days=1)

    df_conv_mode = spark.sql(f"""
        SELECT
            get_json_object(response.result, '$.conversation_id') AS conversation_id,
            CASE request_params.conversation_type
                WHEN 'NORMAL' THEN 'chat'
                WHEN 'DEEP_RESEARCH' THEN 'agent'
            END AS conversation_mode
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name = 'createConversation'
          AND request_params.space_id = '{SPACE_ID}'
          AND request_params.conversation_type IS NOT NULL
          AND event_date >= '{earliest_date}'
    """)

    # Default NULL → 'chat': the implicit genieStartConversationMessage path is always
    # normal chat mode. Deep Research requires explicit createConversation (which IS logged).
    df_new = (
        df_new.alias("d")
        .join(df_conv_mode.alias("m"), F.col("d.conversation_id") == F.col("m.conversation_id"), "left")
        .select("d.*", F.coalesce(F.col("m.conversation_mode"), F.lit("chat")).alias("conversation_mode"))
        .withColumn("question_time", F.from_unixtime(F.col("question_timestamp") / 1000))
        .withColumn("response_time", F.from_unixtime(F.col("response_timestamp") / 1000))
        .withColumn("ingested_at", F.current_timestamp())
    )

    display(df_new.select(
        "conversation_id", "conversation_mode", "user_question",
        "response_type", "question_time",
        "total_duration_sec", "pre_execution_sec", "query_execution_sec", "post_execution_sec"
    ))

In [0]:
# Ensure catalog and schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Create table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FULL_TABLE_NAME} (
    space_id STRING NOT NULL,
    conversation_id STRING NOT NULL,
    conversation_title STRING,
    message_id STRING NOT NULL,
    user_id BIGINT COMMENT 'Databricks user ID of the person who asked the question',
    user_question STRING,
    status STRING,
    response_type STRING COMMENT 'Type of response: sql_query, text_response, error, in_progress',
    conversation_mode STRING COMMENT 'Genie mode: chat (NORMAL) or agent (DEEP_RESEARCH), from audit log',
    question_timestamp BIGINT,
    response_timestamp BIGINT,
    total_duration_sec DOUBLE COMMENT 'Total time from user question to final response (seconds)',
    pre_execution_sec DOUBLE COMMENT 'AI generation + filtering + planning before SQL execution (seconds)',
    query_execution_sec DOUBLE COMMENT 'SQL warehouse query execution time (seconds)',
    post_execution_sec DOUBLE COMMENT 'Result processing + summarization after SQL completes (seconds)',
    sql_statement_id STRING,
    generated_sql STRING,
    question_time TIMESTAMP,
    response_time TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA
COMMENT 'Incremental log of Genie space conversation durations with stage breakdown: total = pre_execution + query_execution + post_execution'
""")

# Add new columns to existing tables (idempotent)
for col_def in [
    ("user_id", "BIGINT", "Databricks user ID of the person who asked the question"),
    ("response_type", "STRING", "Type of response: sql_query, text_response, error, in_progress"),
    ("post_execution_sec", "DOUBLE", "Result processing + summarization after SQL completes (seconds)"),
    ("conversation_mode", "STRING", "Genie mode: chat (NORMAL) or agent (DEEP_RESEARCH), from audit log"),
]:
    try:
        spark.sql(f"ALTER TABLE {FULL_TABLE_NAME} ADD COLUMN {col_def[0]} {col_def[1]} COMMENT '{col_def[2]}'")
        print(f"  Added column: {col_def[0]}")
    except Exception as e:
        if "already exists" in str(e).lower() or "FIELDS_ALREADY_EXISTS" in str(e):
            pass  # Column already exists
        else:
            print(f"  Warning adding {col_def[0]}: {e}")

print(f"Table {FULL_TABLE_NAME} ready.")

In [0]:
from delta.tables import DeltaTable

if timing_records:
    # Use MERGE for incremental upsert — keyed on (space_id, conversation_id, message_id)
    target = DeltaTable.forName(spark, FULL_TABLE_NAME)

    (
        target.alias("t")
        .merge(
            df_new.alias("s"),
            """
            t.space_id = s.space_id
            AND t.conversation_id = s.conversation_id
            AND t.message_id = s.message_id
            """,
        )
        .whenMatchedUpdate(
            set={
                "user_id": "s.user_id",
                "status": "s.status",
                "response_type": "s.response_type",
                "conversation_mode": "s.conversation_mode",
                "response_timestamp": "s.response_timestamp",
                "total_duration_sec": "s.total_duration_sec",
                "pre_execution_sec": "s.pre_execution_sec",
                "query_execution_sec": "s.query_execution_sec",
                "post_execution_sec": "s.post_execution_sec",
                "sql_statement_id": "s.sql_statement_id",
                "generated_sql": "s.generated_sql",
                "response_time": "s.response_time",
                "ingested_at": "s.ingested_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    row_count = spark.table(FULL_TABLE_NAME).count()
    print(f"Merge complete. Table now has {row_count} rows.")
else:
    print("No new records to merge.")

In [0]:
# Overall summary
display(
    spark.sql(f"""
    SELECT
        space_id,
        COUNT(*) AS total_messages,
        COUNT(DISTINCT conversation_id) AS total_conversations,
        COUNT(DISTINCT user_id) AS distinct_users,
        ROUND(AVG(total_duration_sec), 2) AS avg_total_sec,
        ROUND(PERCENTILE(total_duration_sec, 0.5), 2) AS p50_total_sec,
        ROUND(PERCENTILE(total_duration_sec, 0.95), 2) AS p95_total_sec,
        MIN(question_time) AS earliest_question,
        MAX(question_time) AS latest_question
    FROM {FULL_TABLE_NAME}
    GROUP BY space_id
    """)
)

# Breakdown by response type: total = pre_execution + query_execution + post_execution
display(
    spark.sql(f"""
    SELECT
        response_type,
        COUNT(*) AS message_count,
        ROUND(AVG(total_duration_sec), 2) AS avg_total_sec,
        ROUND(AVG(pre_execution_sec), 2) AS avg_pre_exec_sec,
        ROUND(AVG(query_execution_sec), 2) AS avg_query_exec_sec,
        ROUND(AVG(post_execution_sec), 2) AS avg_post_exec_sec,
        ROUND(AVG(COALESCE(pre_execution_sec, 0) + COALESCE(query_execution_sec, 0) + COALESCE(post_execution_sec, 0)), 2) AS avg_sum_of_stages_sec,
        ROUND(PERCENTILE(total_duration_sec, 0.5), 2) AS p50_total_sec,
        ROUND(PERCENTILE(total_duration_sec, 0.95), 2) AS p95_total_sec
    FROM {FULL_TABLE_NAME}
    GROUP BY response_type
    ORDER BY message_count DESC
    """)
)